# RetinaAI · Five-class research model training
**Research only. Not a clinically validated medical device.** Review KAGGLE_TRAINING_GUIDE.md and dataset permissions before running. The committed dummy model is untrained. This notebook trains a separate real research model; no medical performance is claimed before execution and independent evaluation. Enable an authorized Kaggle GPU and attach an appropriately licensed APTOS-style dataset.
The API contract is raw float32 RGB 0–255, 224×224 center-cropped input, and five softmax outputs. Normalization is embedded in EfficientNet. The notebook does not change the frontend.

In [ ]:
from pathlib import Path
import hashlib, json, warnings
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import tensorflow as tf
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, cohen_kappa_score

DATA_ROOT = Path('/kaggle/input/aptos2019-blindness-detection')
LABELS_FILE = DATA_ROOT / 'train.csv'
IMAGE_DIR = DATA_ROOT / 'train_images'
IMAGE_COLUMN, LABEL_COLUMN, IMAGE_EXTENSION = 'id_code', 'diagnosis', '.png'
OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_VERSION = 'research-effnetb0-v1'
CLASS_LABELS = ['No DR', 'Mild DR', 'Moderate DR', 'Severe DR', 'Proliferative DR']
SEED, SIZE, BATCH_SIZE = 2026, 224, 16
WARMUP_EPOCHS, FINETUNE_EPOCHS = 5, 10
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__, 'GPUs:', len(tf.config.list_physical_devices('GPU')))
if not LABELS_FILE.is_file():
    raise FileNotFoundError('Attach an authorized dataset and update DATA_ROOT to its Kaggle mount.')


## Inspect labels and data provenance
Check licensing, label provenance, duplicate/near-duplicate images, and clinical grading definitions before training. Do not upload datasets or identifying manifests to public repositories. Patient groups are used if patient_id exists; otherwise image-level splitting is disclosed as a limitation.

In [ ]:
frame = pd.read_csv(LABELS_FILE)
assert {IMAGE_COLUMN, LABEL_COLUMN}.issubset(frame.columns), 'Required label columns are missing.'
assert not frame[[IMAGE_COLUMN, LABEL_COLUMN]].isnull().any().any(), 'Missing IDs or labels.'
assert not frame[IMAGE_COLUMN].duplicated().any(), 'Duplicate image identifiers must be resolved.'
numeric_labels = pd.to_numeric(frame[LABEL_COLUMN], errors='raise')
assert np.equal(numeric_labels, np.floor(numeric_labels)).all(), 'Labels must be integers.'
frame['label'] = numeric_labels.astype(int)
assert set(frame['label']) == set(range(5)), 'All five severity labels 0–4 are required.'
frame['path'] = frame[IMAGE_COLUMN].map(lambda value: str(IMAGE_DIR / (str(value) + IMAGE_EXTENSION)))
assert frame['path'].map(lambda path: Path(path).is_file()).all(), 'Some images are missing.'
print(frame.groupby('label').size().rename(index=dict(enumerate(CLASS_LABELS))))


In [ ]:
patient_level = 'patient_id' in frame.columns
if patient_level:
    assert frame['patient_id'].notna().all(), 'Patient IDs must not be missing.'
    groups = frame['patient_id'].astype(str)
    first = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    remaining_idx, test_idx = next(first.split(frame, frame['label'], groups))
    remaining = frame.iloc[remaining_idx].copy()
    second = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED + 1)
    train_idx, val_idx = next(second.split(remaining, remaining['label'], remaining['patient_id'].astype(str)))
    train, val, test = remaining.iloc[train_idx].copy(), remaining.iloc[val_idx].copy(), frame.iloc[test_idx].copy()
    assert set(train['patient_id']).isdisjoint(val['patient_id'])
    assert set(train['patient_id']).isdisjoint(test['patient_id'])
    assert set(val['patient_id']).isdisjoint(test['patient_id'])
else:
    warnings.warn('No patient IDs: image-level splitting cannot establish patient independence. Recover verified groups where possible and disclose this limitation.')
    remaining, test = train_test_split(frame, test_size=0.2, stratify=frame['label'], random_state=SEED)
    train, val = train_test_split(remaining, test_size=0.25, stratify=remaining['label'], random_state=SEED + 1)
for name, split in [('train', train), ('validation', val), ('test', test)]:
    assert set(split['label']) == set(range(5)), f'{name} is missing a class; review the split.'
    print(name, len(split), split['label'].value_counts().sort_index().to_dict())
assert set(train['path']).isdisjoint(val['path']) and set(train['path']).isdisjoint(test['path'])
assert set(val['path']).isdisjoint(test['path'])
manifest = pd.concat([part.assign(split=name) for name, part in [('train', train), ('validation', val), ('test', test)]])
manifest.to_csv(OUTPUT_DIR / 'split_manifest.csv', index=False)
class_weights = dict(enumerate(compute_class_weight(class_weight='balanced', classes=np.arange(5), y=train['label'])))


In [ ]:
def pillow_preprocess(path):
    if isinstance(path, np.ndarray):
        path = path.item()
    if isinstance(path, bytes):
        path = path.decode('utf-8')
    with Image.open(path) as source:
        image = ImageOps.exif_transpose(source).convert('RGB')
        image = ImageOps.fit(image, (SIZE, SIZE), method=Image.Resampling.LANCZOS)
        return np.asarray(image, dtype=np.float32)

def decode_row(path, label):
    pixels = tf.numpy_function(pillow_preprocess, [path], tf.float32)
    pixels.set_shape((SIZE, SIZE, 3))
    return pixels, tf.cast(label, tf.int32)

def dataset(split, training=False):
    result = tf.data.Dataset.from_tensor_slices((split['path'].to_numpy(), split['label'].to_numpy(dtype=np.int32)))
    if training:
        result = result.shuffle(len(split), seed=SEED, reshuffle_each_iteration=True)
    result = result.map(decode_row, num_parallel_calls=tf.data.AUTOTUNE, deterministic=True)
    return result.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds, val_ds, test_ds = dataset(train, True), dataset(val), dataset(test)
sample, _ = next(iter(train_ds))
assert sample.shape[1:] == (224, 224, 3) and float(tf.reduce_max(sample)) <= 255
print('Raw RGB range; no external /255 normalization:', float(tf.reduce_min(sample)), float(tf.reduce_max(sample)))


In [ ]:
inputs = tf.keras.Input((SIZE, SIZE, 3), name='rgb_0_to_255')
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.025, fill_mode='reflect', seed=SEED),
    tf.keras.layers.RandomZoom(0.05, fill_mode='reflect', seed=SEED + 1),
], name='training_only_augmentation')
augmented = augmentation(inputs)
backbone = tf.keras.applications.EfficientNetB0(include_top=False, weights='imagenet', input_tensor=augmented)
backbone.trainable = False
features = tf.keras.layers.GlobalAveragePooling2D()(backbone.output)
features = tf.keras.layers.Dropout(0.3)(features)
outputs = tf.keras.layers.Dense(5, activation='softmax', name='severity_scores')(features)
model = tf.keras.Model(inputs, outputs, name='retinaai_research_effnetb0')
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

def callbacks():
    return [tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-7),
            tf.keras.callbacks.TerminateOnNaN()]


In [ ]:
warmup = model.fit(train_ds, validation_data=val_ds, epochs=WARMUP_EPOCHS, class_weight=class_weights, callbacks=callbacks())
backbone.trainable = True
for layer in backbone.layers[:-30]:
    layer.trainable = False
for layer in backbone.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
finetune = model.fit(train_ds, validation_data=val_ds, epochs=FINETUNE_EPOCHS, class_weight=class_weights, callbacks=callbacks())


## Held-out evaluation
Accuracy alone is inadequate for medical screening research. Report per-class sensitivity, macro/weighted precision/recall/F1, confusion matrix, QWK, and a pre-specified research threshold. This template does not establish clinical validity. Do not tune the model or thresholds on test results. Add patient-level confidence intervals, calibration assessment, external validation, and subgroup audits in a reviewed study protocol.

In [ ]:
scores = model.predict(test_ds, verbose=1)
truth = test['label'].to_numpy()
predictions = scores.argmax(axis=1)
assert scores.shape == (len(test), 5) and np.isfinite(scores).all()
assert np.allclose(scores.sum(axis=1), 1, atol=1e-4)
matrix = confusion_matrix(truth, predictions, labels=range(5))
sensitivity = np.divide(matrix.diagonal(), matrix.sum(axis=1), out=np.zeros(5, dtype=float), where=matrix.sum(axis=1) != 0)
report = classification_report(truth, predictions, labels=range(5), target_names=CLASS_LABELS, output_dict=True, zero_division=0)
referable_true, referable_pred = truth >= 2, predictions >= 2
ref_matrix = confusion_matrix(referable_true, referable_pred, labels=[False, True])
tn, fp, fn, tp = ref_matrix.ravel()
evaluation = {'accuracy': float(accuracy_score(truth, predictions)),
              'quadratic_weighted_kappa': float(cohen_kappa_score(truth, predictions, weights='quadratic')),
              'classification_report': report, 'confusion_matrix': matrix.tolist(),
              'per_class_sensitivity': dict(zip(CLASS_LABELS, sensitivity.tolist())),
              'research_grade_ge_2_sensitivity': float(tp / (tp + fn)) if tp + fn else None,
              'research_grade_ge_2_specificity': float(tn / (tn + fp)) if tn + fp else None,
              'patient_level_split': patient_level, 'clinically_validated': False}
(OUTPUT_DIR / 'evaluation.json').write_text(json.dumps(evaluation, indent=2))
np.savetxt(OUTPUT_DIR / 'confusion_matrix.csv', matrix, fmt='%d', delimiter=',')
print(json.dumps(evaluation, indent=2))


In [ ]:
artifact = OUTPUT_DIR / 'retina_dr_model.keras'
model.save(artifact)
reloaded = tf.keras.models.load_model(artifact, compile=False, safe_mode=True)
assert reloaded.input_shape == (None, 224, 224, 3) and reloaded.output_shape == (None, 5)
verification_batch, _ = next(iter(test_ds))
before = np.asarray(model(verification_batch, training=False))
after = np.asarray(reloaded(verification_batch, training=False))
np.testing.assert_allclose(before, after, rtol=1e-4, atol=1e-5)
assert np.isfinite(after).all() and (after >= 0).all() and np.allclose(after.sum(axis=1), 1, atol=1e-4)
metadata = {'model_type': 'TRAINED', 'model_version': MODEL_VERSION, 'model_name': model.name,
            'class_labels': CLASS_LABELS, 'input_shape': [None, 224, 224, 3], 'channels': 'RGB',
            'input_range': [0, 255], 'dtype': 'float32', 'normalization': 'embedded EfficientNet rescaling',
            'resize': 'Pillow ImageOps.fit center crop 224x224 LANCZOS after EXIF orientation',
            'tensorflow_version': tf.__version__, 'patient_level_split': patient_level,
            'sha256': hashlib.sha256(artifact.read_bytes()).hexdigest(), 'clinically_validated': False,
            'usage': 'Research only. Not a clinical diagnosis or treatment recommendation.'}
(OUTPUT_DIR / 'model_metadata.json').write_text(json.dumps(metadata, indent=2))
print('Verified export:', artifact.name)
print('Backend configuration: MODEL_TYPE=TRAINED; MODEL_VERSION=' + MODEL_VERSION + '; MODEL_PATH=app/ml/retina_dr_model.keras')


## Integration checkpoint
Download the model and metadata. Place the model in backend/app/ml, set MODEL_PATH, MODEL_TYPE=TRAINED, MODEL_VERSION, and SEED_DEMO=false, then restart FastAPI. Check /api/health and send a synthetic test image. The loader rejects incompatible shapes and refuses to label the committed demo network as trained. The frontend consumes the same API and automatically changes model labels. Trained softmax scores are not clinical confidence. Real patient use still requires authentication, role-based access, security controls, independent clinical validation, and applicable regulatory review.